# Example 1

The original mixed-geometry scene: a CST transmitter aimed along +x and an isotropic receiver.

Run this notebook with the repository's `.venv` kernel after installing `raytracing`.
The package includes the CST patch, the C++ numerical solver, Matplotlib and PyVista. Positions and
lengths are in metres; frequencies are in Hz. Plots and scene images display inline.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
import raytracing as rt

print(f"RayTracing {rt.__version__}")

## Simulation settings

In [ ]:
config = rt.SolverConfig(
    frequency_hz=77e9,
    ray_count=20_000,
    max_reflections=8,
    max_distance=20.0,
)
output = Path("results/example1")
output.mkdir(parents=True, exist_ok=True)

## Scene and antennas

In [ ]:
x, y, z = np.eye(3)
scene = rt.Scene()
scene.add(rt.Rectangle([1.5, 0, 0], y, z, 3, 1))
scene.add(rt.Rectangle([-1.5, 0, 0], y, z, 3, 1))
scene.add(rt.Rectangle([0, -3, 0], x, z, 3, 1))
scene.add(rt.Rectangle([0, 3, 0], x, z, 3, 1))
scene.add(rt.Sphere([0.7, -1.2, 0], 0.3))
scene.add(rt.Disk([-0.7, -1.3, 0], y, 0.35))
axis = np.array([0.2, 0, 1.0])
axis /= np.linalg.norm(axis)
scene.add(rt.Cylinder([0.6, 2, 0], axis, 0.25, 0.7, capped=True))
scene.add(rt.Triangle([-1, 0, 0.8], [0, 0, 0.8], [-0.5, 1, 0.8]))

patch = rt.load_farfield(rt.example_pattern())  # CST positive-time import by default.
patch.rotate(y, -90.0)  # Native -z beam rotated toward horizontal +x.
tx = rt.Transmitter([0, 0, 0], patch)
rx = rt.Receiver([-0.5, 1.4, 0], rt.Isotropic())
len(scene)

## Solve and inspect the returns

In [ ]:
result = rt.solve(scene, tx, rx, config)
{
    "launched_rays": result.launched_rays,
    "refined_paths": len(result.rays),
    "paths_with_fields": len(result.response_ray_indices),
    "impulse_taps": len(result.impulse_response.taps),
}

In [ ]:
# Each ray exposes NumPy vertices, complex fields, geometry and numerical diagnostics.
[
    {
        "path": i,
        "reflections": len(ray.refinement.geometry.reflections),
        "length_m": ray.refinement.geometry.receiver.path_distance,
        "field_status": ray.spreading.status.name,
        "received_magnitude": abs(ray.received_coefficient),
    }
    for i, ray in enumerate(result.rays[:20])
]

## Save and visualize

In [ ]:
rt.save_h5(result, output / "simulation.h5")
rt.save_impulse_csv(result.impulse_response, output / "impulse_response.csv")
output.resolve()

In [ ]:
figure = rt.plot(result.impulse_response, output / "impulse_response.png")
display(figure)
plt.close(figure)

In [ ]:
viewer = rt.visualize(result)
viewer.screenshot(str(output / "scene.png"))
display(Image(filename=str(output / "scene.png")))
viewer.close()

## Reload without retracing

The HDF5 file embeds both antenna patterns and the complete simulation. `rt.visualize(saved)` returns a PyVista Plotter for camera and scene customization. Use `viewer.show(jupyter_backend='static')` to display it inline, or `viewer.screenshot(path)` to save it, then `viewer.close()` when finished.

In [ ]:
saved = rt.load_h5(output / "simulation.h5")
rt.frequency_response(saved.impulse_response)  # Complex channel response at the carrier.